<a href="https://colab.research.google.com/github/aakriti1318/llm-inference-sandbox/blob/main/LLM%20Benchmarking%20Playground/LLM_Benchmarking_Playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Benchmarking Playground: vLLM vs. Ollama Across Industry Harnesses

> Running on T4 GPU

This notebook serves as a hands-on learning environment for benchmarking Large Language Models using local inference servers and popular evaluation frameworks. It explores performance, accuracy, and agentic capability through four distinct benchmarking pipelines:

* **lm-evaluation-harness (via vLLM):** Evaluates academic task accuracy (e.g., HellaSwag) using a high-throughput OpenAI-compatible API backend.
* **EvalPlus (via vLLM):** Rigorously tests code-generation correctness and strict functional pass rates (e.g., HumanEval) via local model serving.
* **Custom Benchmarking (via vLLM):** Implements personalized, domain-specific evaluation scripts targeting custom datasets and latency metrics.
* **Harbor (via Ollama):** Tests agentic workflows and terminal-based task execution capabilities using Ollama's lightweight local runner.

## vLLM Installation

In [1]:
!uv init

Initialized project `content`


In [ ]:
!uv venv

In [ ]:
!uv pip install vllm --torch-backend=auto

In [ ]:
!uv pip uninstall torchaudio

In [ ]:
!uv pip install --force-reinstall torch --index-url https://download.pytorch.org/whl/cu132

## Start the vLLM server

In [46]:
!nohup uv run vllm serve Qwen/Qwen2.5-Coder-0.5B-Instruct --host 127.0.0.1 --port 8000 --gpu-memory-utilization 0.2 > vllm.log 2>&1 &

In [48]:
!cat vllm.log

(APIServer pid=35410) INFO 08-27 20:04:51 [api_utils.py:333] 
(APIServer pid=35410) INFO 08-27 20:04:51 [api_utils.py:333]        █     █     █▄   ▄█
(APIServer pid=35410) INFO 08-27 20:04:51 [api_utils.py:333]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.28.0
(APIServer pid=35410) INFO 08-27 20:04:51 [api_utils.py:333]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-Coder-0.5B-Instruct
(APIServer pid=35410) INFO 08-27 20:04:51 [api_utils.py:333]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=35410) INFO 08-27 20:04:51 [api_utils.py:333] 
(APIServer pid=35410) INFO 08-27 20:04:51 [api_utils.py:272] non-default args: {'model_tag': 'Qwen/Qwen2.5-Coder-0.5B-Instruct', 'host': '127.0.0.1', 'model': 'Qwen/Qwen2.5-Coder-0.5B-Instruct', 'gpu_memory_utilization': 0.2}
(APIServer pid=35410) INFO 08-27 20:04:52 [model.py:672] Resolved architecture: Qwen2ForCausalLM
(APIServer pid=35410) WARNING 08-27 20:04:52 [model.py:2246] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat1

## Harness 1 - LLM-Evaluation-Harness

Github Link - https://github.com/eleutherai/lm-evaluation-harness

This project provides a unified framework to test generative language models on a large number of different evaluation tasks.

In [20]:
!uv pip install "lm_eval[hf,vllm,api]"

Using Python 3.13.15 environment at: /usr
Checked 1 package in 105ms


In [ ]:
!uv run lm-eval ls tasks
# list available tasks

In [ ]:
!uv run lm-eval ls tasks | grep hellaswag
# Tasks to predict the ending of stories or scenarios, testing comprehension and creativity.

* `--model` - Specifies the evaluation backend wrapper to use (local-completions tells lm-evaluation-harness to interface with a local API endpoint instead of loading raw weights into memory locally).

* `--model_args` - A comma-separated string of configuration options passed directly to the model wrapper.

* `model`=Qwen/Qwen2.5-Coder-0.5B-Instruct - The identifier of the model being evaluated (used by the tokenizer and logging).

* `base_url`=http://localhost:8000/v1/completions - The URL of the local vLLM server endpoint that accepts requests.

* `tokenizer`=Qwen/Qwen2.5-Coder-0.5B-Instruct - The Hugging Face repo ID used to load the correct tokenizer for encoding evaluation contexts.

* `tokenizer_backend`=huggingface - Directs the framework to use the local Hugging Face tokenizer implementation.

* `tokenized_requests`=False - Specifies whether raw token IDs or text strings should be sent across the API payload.

* `--tasks` - Defines the benchmark dataset and evaluation task to run (hellaswag in this case).

* `--limit` - Restricts the evaluation to a subset of samples (e.g., first 20 examples) for quick sanity checks.

In [38]:
!uv run lm-eval run --model local-completions --model_args "model=Qwen/Qwen2.5-Coder-0.5B-Instruct,base_url=http://localhost:8000/v1/completions,tokenizer=Qwen/Qwen2.5-Coder-0.5B-Instruct,tokenizer_backend=huggingface,tokenized_requests=False" --tasks hellaswag --limit 20

2026-08-27:19:07:26 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-08-27:19:07:36 INFO     [_cli.run:388] Selected Tasks: ['hellaswag']
2026-08-27:19:07:36 WARNING  [evaluator:184] pretrained=None appears to be an instruct or chat variant but chat template is not applied. Recommend setting `apply_chat_template`
        (optionally `fewshot_as_multiturn`).
2026-08-27:19:07:38 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-08-27:19:07:38 INFO     [evaluator:239] Initializing local-completions model, with arguments: {'model': 'Qwen/Qwen2.5-Coder-0.5B-Instruct', 'base_url': 'http://localhost:8000/v1/completions', 'tokenizer': 'Qwen/Qwen2.5-Coder-0.5B-Instruct', 'tokenizer_backend': 'huggingface', 'tokenized_requests': False}
2026-08-27:19:07:38 INFO     [models.api_models:179] Using max length 20

In [ ]:
# !cat vllm.log

In [48]:
!uv run lm-eval run --model local-completions --model_args "model=Qwen/Qwen2.5-Coder-0.5B-Instruct,base_url=http://localhost:8000/v1/completions,tokenizer=Qwen/Qwen2.5-Coder-0.5B-Instruct,tokenizer_backend=huggingface,tokenized_requests=False" --tasks hellaswag --limit 100

2026-08-27:19:14:36 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-08-27:19:14:46 INFO     [_cli.run:388] Selected Tasks: ['hellaswag']
2026-08-27:19:14:46 WARNING  [evaluator:184] pretrained=None appears to be an instruct or chat variant but chat template is not applied. Recommend setting `apply_chat_template`
        (optionally `fewshot_as_multiturn`).
2026-08-27:19:14:48 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-08-27:19:14:48 INFO     [evaluator:239] Initializing local-completions model, with arguments: {'model': 'Qwen/Qwen2.5-Coder-0.5B-Instruct', 'base_url': 'http://localhost:8000/v1/completions', 'tokenizer': 'Qwen/Qwen2.5-Coder-0.5B-Instruct', 'tokenizer_backend': 'huggingface', 'tokenized_requests': False}
2026-08-27:19:14:48 INFO     [models.api_models:179] Using max length 20

* For 20 example - Got the accuracy of 0.3 (not good) and standard error 0.1051
* For 100 example - Got the accuracy of 0.4 (better) and standard error 0.0492


## Harness 2 - Evalplus

Github Link - https://github.com/evalplus/evalplus

EvalPlus is a rigorous evaluation framework for LLM4Code, with:

- HumanEval+: 80x more tests than the original HumanEval!
- MBPP+: 35x more tests than the original MBPP!
- EvalPerf: evaluating the efficiency of LLM-generated code!
- Framework: our packages/images/tools can easily and safely evaluate LLMs on above benchmarks.


In [ ]:
!uv add evalplus

In [8]:
!uv run evalplus.evaluate --model Qwen/Qwen2.5-Coder-0.5B-Instruct --dataset humaneval --backend openai --base_url http://localhost:8000/v1 --greedy

Greedy decoding ON (--greedy): setting bs=1, n_samples=1, temperature=0
Initializing a decoder model: Qwen/Qwen2.5-Coder-0.5B-Instruct ...
Sanitized code outputs will be saved to evalplus_results/humaneval/Qwen--Qwen2.5-Coder-0.5B-Instruct_openai_temp_0.0.jsonl
Raw outputs will be saved to evalplus_results/humaneval/Qwen--Qwen2.5-Coder-0.5B-Instruct_openai_temp_0.0.raw.jsonl
Codegen: HumanEval/0 @ Qwen/Qwen2.5-Coder-0.5B-Instruct
Codegen: HumanEval/1 @ Qwen/Qwen2.5-Coder-0.5B-Instruct
Codegen: HumanEval/2 @ Qwen/Qwen2.5-Coder-0.5B-Instruct
Codegen: HumanEval/3 @ Qwen/Qwen2.5-Coder-0.5B-Instruct
Codegen: HumanEval/4 @ Qwen/Qwen2.5-Coder-0.5B-Instruct
Codegen: HumanEval/5 @ Qwen/Qwen2.5-Coder-0.5B-Instruct
Codegen: HumanEval/6 @ Qwen/Qwen2.5-Coder-0.5B-Instruct
Codegen: HumanEval/7 @ Qwen/Qwen2.5-Coder-0.5B-Instruct
Codegen: HumanEval/8 @ Qwen/Qwen2.5-Coder-0.5B-Instruct
Codegen: HumanEval/9 @ Qwen/Qwen2.5-Coder-0.5B-Instruct
Codegen: HumanEval/10 @ Qwen/Qwen2.5-Coder-0.5B-Instruct
Codeg

## Harness 3 - Custom Benchmarking

#### Create the my_benchmark.jsonl to contain the examples

In [49]:
%%writefile my_benchmark.jsonl
{"question": "What is 2+2?", "choices": ["3", "4", "5", "6"], "answer": 1}
{"question": "Capital of France?", "choices": ["Berlin", "Paris", "Rome", "Madrid"], "answer": 1}
{"question": "Which is a mammal?", "choices": ["Shark", "Eagle", "Whale", "Trout"], "answer": 2}

Writing my_benchmark.jsonl


#### Create the my_benchmark.yaml to contain the benchmarking logic

In [50]:
%%writefile my_benchmark.yaml
task: my_benchmark
dataset_path: json
dataset_kwargs:
  data_files:
    test: my_benchmark.jsonl
test_split: test
output_type: multiple_choice
doc_to_text: "{{question}}\nAnswer:"
doc_to_choice: "{{choices}}"
doc_to_target: answer
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: true
metadata:
  version: 1

Writing my_benchmark.yaml


In [51]:
!uv run lm-eval run \
    --model local-completions \
    --model_args "model=Qwen/Qwen2.5-Coder-0.5B-Instruct,base_url=http://localhost:8000/v1/completions,tokenizer=Qwen/Qwen2.5-Coder-0.5B-Instruct,tokenizer_backend=huggingface,tokenized_requests=False" \
    --tasks my_benchmark \
    --include_path . \
    --limit 20

2026-08-27:20:12:14 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-08-27:20:12:24 INFO     [_cli.run:387] Including path: .
2026-08-27:20:12:24 INFO     [_cli.run:388] Selected Tasks: ['my_benchmark']
2026-08-27:20:12:24 WARNING  [evaluator:184] pretrained=None appears to be an instruct or chat variant but chat template is not applied. Recommend setting `apply_chat_template`
        (optionally `fewshot_as_multiturn`).
2026-08-27:20:12:27 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-08-27:20:12:27 INFO     [evaluator:239] Initializing local-completions model, with arguments: {'model': 'Qwen/Qwen2.5-Coder-0.5B-Instruct', 'base_url': 'http://localhost:8000/v1/completions', 'tokenizer': 'Qwen/Qwen2.5-Coder-0.5B-Instruct', 'tokenizer_backend': 'huggingface', 'tokenized_requests': False}
2026-08

## Ollama Installation

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Start the Ollama Server in the Background

In [20]:
import subprocess
import threading
import time

def run_ollama_server():
    # Starts the server process
    subprocess.Popen(["ollama", "serve"])

# Start the server in a background thread
server_thread = threading.Thread(target=run_ollama_server, daemon=True)
server_thread.start()

# Give the server a few seconds to initialize
time.sleep(5)
print("Ollama server started in the background!")

Ollama server started in the background!


In [37]:
!nohup ollama serve > ollama.log 2>&1 &

In [38]:
!cat ollama.log

time=2026-08-27T19:53:13.892Z level=INFO source=routes.go:1951 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* 

In [39]:
!ollama pull qwen2.5:0.5b

In [40]:
!ollama list

NAME            ID              SIZE      MODIFIED      
qwen2.5:0.5b    a8b0c5157701    397 MB    2 seconds ago    


## Harness 4 - Harbor

Github Link - https://github.com/harbor-framework/harbor

Harbor is a framework from the creators of Terminal-Bench for evaluating and optimizing agents and language models. You can use Harbor to:

- Evaluate arbitrary agents like Claude Code, OpenHands, Codex CLI, and more.
- Build and share your own benchmarks and environments.
- Conduct experiments in thousands of environments in parallel through providers like Daytona, Modal, LangSmith, Blaxel, Novita Sandbox, and Tensorlake.
- Generate rollouts for RL optimisation.

In [ ]:
!uv add harbor

* `uv run` - Executes the command inside the project's managed virtual environment using the `uv` package runner.
* `harbor run` - The core command to initiate a benchmark execution run using the Harbor evaluation framework.
* `-d terminal-bench/terminal-bench-2-1` - Specifies the benchmark dataset/task collection to evaluate against (Terminal-Bench version 2.1).
* `-a terminus-2` - Selects the agent or harness backend adapter to use for managing the benchmark environment (`terminus-2`).
* `-m "ollama_chat/qwen2.5-coder:0.5b"` - Sets the model identifier and provider interface to be tested (in this case, the Qwen 2.5 Coder 0.5B model hosted via Ollama's chat endpoint).
* `--ak api_base=http://localhost:11434` - Passes additional keyword arguments to the model adapter, pointing it to your local Ollama server address.
* `-n 1` - Limits the run to a single test sample or concurrent worker instance for a quick test execution.

In [ ]:
!uv run harbor run -d terminal-bench/terminal-bench-2-1 -a terminus-2 -m "ollama_chat/qwen2.5-coder:0.5b" --ak api_base=http://localhost:11434 -n 1

* `-t terminal-bench/filter-js-from-html` - Replaces the dataset flag (`-d`) to target a single specific evaluation task (`filter-js-from-html`) instead of an entire benchmark suite.

In [ ]:
!uv run harbor run -t terminal-bench/filter-js-from-html -a terminus-2 -m "ollama_chat/qwen2.5-coder:0.5b" --ak api_base=http://localhost:11434 -n 1